# EDA Implied Repo Rate

# Setup

## Imports

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.tools.sm_exceptions import InterpolationWarning
from statsmodels.tsa.stattools import adfuller, kpss

from thesis_project import (
    config as global_config,
)
from thesis_project.importance_mi import (
    data_pipeline,
)

## Paths

In [ ]:
TICKER = "fbtp"
NOTEBOOK = "eda-irr"

# Lobs directories
LOB_DIR = global_config.RAW_DIR / TICKER
LOB_FUT_DIR = LOB_DIR / "futures"
LOB_CTD_DIR = LOB_DIR / "ctd"

# Output directories
if NOTEBOOK is not None:
    OUT_FIG_DIR = global_config.FIG_DIR / TICKER / NOTEBOOK
    OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)
    OUT_TAB_DIR = global_config.TAB_DIR / TICKER / NOTEBOOK
    OUT_TAB_DIR.mkdir(parents=True, exist_ok=True)
    # OUT_INT_DIR = global_config.INT_DIR / TICKER / NOTEBOOK
    # OUT_INT_DIR.mkdir(parents=True, exist_ok=True)
    # OUT_PRO_DIR = global_config.PRO_DIR / TICKER / NOTEBOOK
    # OUT_PRO_DIR.mkdir(parents=True, exist_ok=True)

## Settings

In [ ]:
# Matplotlib global settings
# config.MPL contains a personalized default configuration for matplotlib
plt.rcdefaults()
plt.style.use("seaborn-v0_8-paper")

In [ ]:
# Warnings settings
warnings.simplefilter("always", UserWarning)

In [ ]:
# Sample date for quick analysis
year = "2022"
month = "09"
day = "07"
date_str = f"{year}_{month}_{day}"
fut_filename = f"futures_lob_freq_1s_{date_str}.parquet"
ctd_filename = f"ctd_lob_freq_1s_{date_str}.parquet"

## Utilities

### Plotting

# Main

## Common

In [ ]:
# IRR data
filename = "irr_freq_1s_from_2022_08_01_to_2024_12_30.parquet"
irr_df = pd.read_parquet(global_config.INT_DIR / TICKER / "data-microstructure" / "irr" / filename)
irr_df = irr_df.astype("float32")
irr_df = irr_df.sort_index()

FILTER = True
if FILTER:
    dates_filters_offsets = {"fut_last_trading_days": [5, 5]}
    irr_df = data_pipeline.filter_dates(irr_df, TICKER, dates_filters_offsets)

assert isinstance(irr_df.index, pd.DatetimeIndex)
idx = irr_df.index
tz = idx.tz

IRR_RESAMPLE_FREQS = [
    "1s",
    "5s",
    "10s",
    "15s",
    "30s",
    "1min",
]

In [ ]:
fut_meta_df = pd.read_csv(
    LOB_DIR / "fut_metadata.csv", parse_dates=["Last Trading Date", "Delivery Date"]
)

ctd_switch_df = pd.read_csv(LOB_DIR / "ctd_switch.csv", parse_dates=["CTD Switch Date"])

In [ ]:
# Get filtered relevant dates
start = irr_df.index.min().normalize()
end = irr_df.index.max().normalize()

# Last trading dates of futures
FUT_LAST_TRADING_DATES = pd.to_datetime(fut_meta_df["Last Trading Date"].values)
FUT_LAST_TRADING_DATES = FUT_LAST_TRADING_DATES.tz_localize(tz).normalize()
FUT_LAST_TRADING_DATE_FILTERED = FUT_LAST_TRADING_DATES[
    (FUT_LAST_TRADING_DATES >= start) & (FUT_LAST_TRADING_DATES <= end)
]

# CTD switch dates
CTD_SWITCH_DATES = pd.to_datetime(ctd_switch_df["CTD Switch Date"].values)
CTD_SWITCH_DATES = CTD_SWITCH_DATES.tz_localize(tz).normalize()
CTD_SWITCH_DATES_FILTERED = CTD_SWITCH_DATES[
    (CTD_SWITCH_DATES >= start) & (CTD_SWITCH_DATES <= end)
]

## Distribution

In [ ]:
info = []
for freq in IRR_RESAMPLE_FREQS:
    irr = irr_df["IRR"].resample(freq).last()
    stats = irr.describe()
    stats["skewness"] = irr.skew()
    stats["kurtosis"] = irr.kurtosis()
    stats.name = f"{freq}"
    info.append(stats)
info = pd.DataFrame(info)
info.index.name = "Resample Frequency"

# info.to_csv(OUT_TAB_DIR / "irr_describe_summary.csv")

display(info)

In [ ]:
# Plot IRR distribution
import matplotlib.ticker as mtick

start_date = irr_df.index.min()
end_date = irr_df.index.max()

q_min = 0.01
q_max = 0.99

for freq in IRR_RESAMPLE_FREQS[0:1]:
    plot_data = irr_df["IRR"].resample(freq).last().dropna()

    upper = plot_data.quantile(q_max)
    lower = plot_data.quantile(q_min)
    mask = plot_data.between(lower, upper)
    plot_data = plot_data[mask]

    weights = np.ones_like(plot_data) / len(plot_data)

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.hist(
        plot_data,
        bins=750,
        weights=weights,
        # density=True,
    )

    title = (
        f"IRR Distribution ({freq} Resampled)"
        f"\nFrom {start_date.strftime('%Y-%m-%d')} To {end_date.strftime('%Y-%m-%d')}"
        f"\nQuantiles {q_min * 100:.1f} % - {q_max * 100:.1f} %"
    )
    ax.set_title(title)
    ax.set_xlabel("IRR")
    ax.set_ylabel("Frequency")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.grid(True, alpha=0.2)

    median = plot_data.median()
    mean = plot_data.mean()
    std = plot_data.std()

    ax.axvline(
        median, color="green", linestyle="--", linewidth=0.75, label=f"Median ({median:.4f})"
    )
    ax.axvline(mean, color="red", linestyle="--", linewidth=0.75, label=f"Mean ({mean:.4f})")
    ax.axvline(mean + std, color="orange", linestyle="--", linewidth=0.75, label=r"$\pm$ 1 Std")
    ax.axvline(mean - std, color="orange", linestyle="--", linewidth=0.75)

    ax.legend(loc="upper right")

    # fig.savefig(OUT_FIG_DIR / f"irr_distribution_{freq}.png", dpi=300)

    plt.tight_layout()
    plt.show()

## Time Series

In [ ]:
# Plot IRR timeseries
from matplotlib import dates as mdates

start_date = irr_df.index.min()
end_date = irr_df.index.max()

start_xlim = start_date - pd.Timedelta(days=35)
end_xlim = end_date + pd.Timedelta(days=15)

for freq in IRR_RESAMPLE_FREQS[0:1]:
    plot_data = irr_df["IRR"].resample(freq).last().dropna()

    plot_data = plot_data.resample("1min").last().dropna()

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(
        plot_data,
        label="IRR",
        linewidth=0.5,
    )

    title = (
        f"IRR Time Series ({freq} Resampled)"
        f"\nFrom {start_date.strftime('%Y-%m-%d')} To {end_date.strftime('%Y-%m-%d')}"
    )
    ax.set_title(title)
    ax.set_xlabel("Time")
    ax.set_ylabel("IRR")
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_locator(mdates.MonthLocator(range(1, 13), interval=3, tz=tz))
    ax.set_xlim(start_xlim, end_xlim)

    for i, d in enumerate(FUT_LAST_TRADING_DATE_FILTERED):
        ax.axvline(
            d,
            color="black",
            linestyle="--",
            linewidth=0.5,
            label="Futures Last Trading Date" if i == 0 else None,
        )

    # for i, d in enumerate(CTD_SWITCH_DATES_FILTERED):
    #     ax.axvline(
    #         d,
    #         color='black',
    #         linestyle='--',
    #         linewidth=0.5,
    #         label='CTD Switch Date' if i == 0 else None
    #     )

    ax.axhline(0, color="red", linestyle="--", linewidth=0.5)
    ax.legend(loc="upper left")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # fig.savefig(OUT_FIG_DIR / f"irr_timeseries_{freq}.png", dpi=300)

    plt.tight_layout()
    plt.show()

## Mean Reversion

### Day-wise Stationarity

In [ ]:
def check_stationarity(series, regression):
    adf_stat, adf_pvalue, *_ = adfuller(series, regression=regression)
    kpss_stat, kpss_pvalue, *_ = kpss(series, regression=regression)

    return {
        "ADF": {
            "Statistic": adf_stat,
            "p-value": adf_pvalue,
        },
        "KPSS": {
            "Statistic": kpss_stat,
            "p-value": kpss_pvalue,
        },
    }


STATIONARITY_TESTS = {
    "c": "Constant",
    "ct": "Constant and Trend",
}

In [ ]:
# Perform day-wise stationarity tests on irr
assert isinstance(irr_df.index, pd.DatetimeIndex)
warnings.filterwarnings("ignore", category=InterpolationWarning)

results = []

for date, group in irr_df.groupby(irr_df.index.normalize()):
    row = {"Date": date.date()}
    irr = group["IRR"]

    for freq in IRR_RESAMPLE_FREQS:
        resampled = irr if freq == "1s" else irr.resample(freq).last()
        resampled = resampled.dropna()

        for reg, reg_name in STATIONARITY_TESTS.items():
            out = check_stationarity(resampled, reg)
            row.update(
                {
                    f"ADF Statistic {freq} {reg_name}": out["ADF"]["Statistic"],
                    f"ADF p-value {freq} {reg_name}": out["ADF"]["p-value"],
                    f"KPSS Statistic {freq} {reg_name}": out["KPSS"]["Statistic"],
                    f"KPSS p-value {freq} {reg_name}": out["KPSS"]["p-value"],
                }
            )

    results.append(row)

results_df = pd.DataFrame.from_records(results).set_index("Date")

In [ ]:
path = global_config.INT_DIR / TICKER / "data-microstructure" / "irr"
filename = "irr_stationarity_tests.parquet"

# results_df.to_parquet(path / filename)
results_df = pd.read_parquet(path / filename)
display(results_df.head())

#### Summary

In [ ]:
import re


def to_seconds(s):
    match = re.match(r"(\d+)([a-zA-Z]+)", s)
    value, unit = match.groups()  # type: ignore
    value = int(value)

    multipliers = {
        "s": 1,
        "min": 60,
        "h": 3600,
    }

    return value * multipliers[unit]


summary = []

for reg, reg_name in STATIONARITY_TESTS.items():
    for freq in IRR_RESAMPLE_FREQS:
        adf_col = f"ADF p-value {freq} {reg_name}"
        kpss_col = f"KPSS p-value {freq} {reg_name}"
        adf_ratio = (results_df[adf_col] <= 0.05).mean()
        kpss_ratio = (results_df[kpss_col] > 0.05).mean()
        summary.append(
            {
                "Regression": reg_name,
                "Frequency": freq,
                "ADF Stationary Ratio": f"{adf_ratio * 100:.2f} %",
                "KPSS Stationary Ratio": f"{kpss_ratio * 100:.2f} %",
            }
        )

summary_df = (
    pd.DataFrame(summary)
    .sort_values(
        by=["Regression", "Frequency"],
        key=lambda x: x.map(lambda s: to_seconds(s) if s in IRR_RESAMPLE_FREQS else s),
    )
    .set_index(["Regression", "Frequency"])
)

# summary_df.to_csv(OUT_TAB_DIR / "irr_stationarity_summary.csv")

display(summary_df)

#### p-value Distribution

In [ ]:
import numpy as np
from matplotlib import ticker as mtick

bins = np.linspace(0, 1, 101)

for reg, reg_name in STATIONARITY_TESTS.items():
    n_plots = len(IRR_RESAMPLE_FREQS)
    cols = 3
    rows = (n_plots + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(12, 6), constrained_layout=True)
    axes = axes.flatten()

    for freq, ax in zip(IRR_RESAMPLE_FREQS, axes):
        for test in ["ADF", "KPSS"]:
            plot_data = results_df[f"{test} p-value {freq} {reg_name}"]
            weights = np.ones_like(plot_data) / len(plot_data)
            ax.hist(
                plot_data,
                bins=bins,  # type: ignore
                alpha=0.5,
                label=f"{test}",
                weights=weights,
            )

        ax.set_title(f"Resampled {freq}")
        ax.set_xlabel("p-value")
        ax.set_ylabel("Frequency")
        ax.set_xticks(bins[::5])
        ax.set_xticks(bins[::1], minor=True)
        ax.set_xlim(0, 0.25)
        ax.set_ylim(0, 1.05)
        ax.grid(True, alpha=0.3)
        ax.axvline(0.05, color="red", linestyle="-", linewidth=0.5, label="Significance Level (5%)")
        ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))

    suptitle = (
        f"IRR Daily Stationarity Tests"
        f"\np-value Distributions"
        f"\nRegression Hypothesis: {reg_name}"
    )
    fig.suptitle(suptitle)
    handles, labels = axes.flatten()[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper right")

    fig.savefig(
        OUT_FIG_DIR
        / f"irr_stationarity_pvalue_distribution_{reg_name.replace(' ', '_').lower()}.png",
        dpi=300,
    )

    plt.show()

#### p-value Time Series

In [ ]:
n_plots = len(IRR_RESAMPLE_FREQS)
cols = 3
rows = (n_plots + cols - 1) // cols

for reg, reg_name in STATIONARITY_TESTS.items():
    fig, axes = plt.subplots(rows, cols, figsize=(12, 6), constrained_layout=True)
    axes = axes.flatten()

    for ax, freq in zip(axes, IRR_RESAMPLE_FREQS):
        for test in ["ADF", "KPSS"]:
            plot_data = results_df[f"{test} p-value {freq} {reg_name}"]
            ax.plot(
                plot_data,
                label=f"{test} p-value",
                linewidth=0.5,
                alpha=0.7,
            )

        ax.axhline(0.05, color="red", linestyle="-", linewidth=0.5, label="Significance Level (5%)")

        # for i, d in enumerate(FUT_LAST_TRADING_DATE_FILTERED):
        #     ax.axvline(
        #         d,
        #         color="black",
        #         linestyle="--",
        #         linewidth=0.5,
        #         label="Futures Last Trading Date" if i == 0 else None,
        #         alpha=0.7,
        #     )

        for i, d in enumerate(CTD_SWITCH_DATES_FILTERED):
            ax.axvline(
                d,
                color="gray",
                linestyle="--",
                linewidth=0.5,
                label="CTD Switch Date" if i == 0 else None,
                alpha=0.7,
            )

        title = f"Resampling Frequency: {freq}"
        ax.set_title(title)
        ax.set_xlabel("Date")
        ax.set_ylabel("p-value")
        ax.grid(True, alpha=0.3)

        ax.xaxis.set_major_locator(mdates.MonthLocator(range(1, 13), interval=6, tz=tz))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m", tz=tz))
        ax.xaxis.set_minor_locator(mdates.MonthLocator(range(1, 13), interval=1, tz=tz))
        ax.set_xlim(start_xlim, end_xlim)

    suptitle = (
        f"IRR Daily Stationarity Tests"
        f"\np-value Time Series"
        f"\nRegression Hypothesis: {reg_name}"
    )
    fig.suptitle(suptitle)
    handles, labels = axes.flatten()[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper right")

    fig.savefig(
        OUT_FIG_DIR
        / f"irr_stationarity_pvalue_timeseries_{reg_name.replace(' ', '_').lower()}.png",
        dpi=300,
    )

    plt.show()